In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Project directory
PROJECT_DIR = Path(
    r"C:\India_Vs_West_Indies_T20i_Challenge"
)

# Cleaned dataset path
CLEAN_FILE = (
    PROJECT_DIR
    / "Data"
    / "Clean"
    / "t20i_ball_by_ball_clean.csv"
)

# Load the cleaned dataset
df = pd.read_csv(CLEAN_FILE)

print("Dataset loaded successfully!")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

display(df.head())

Dataset loaded successfully!
Rows: 959
Columns: 24


,Venue,MatchDay,Innings,Batting Team,Bowling Team,Striker,Non-Striker,Bowler,Ball Number,Actual Ball Number,...,Wide,No Ball,Leg Bye,Bye,Wicket,4's,6's,Run_QC_Flag,Extras_QC_Flag,Ball_Number_Label
0,Ekana Cricket Stadium B Ground,1,1,West Indies Cricket Team,India National Cricket team,Shai Hope,Kamil Pooran,Arshdeep Singh,0.1,0.1,...,0,0,0,0,0,0,0,False,False,0.1
1,Ekana Cricket Stadium B Ground,1,1,West Indies Cricket Team,India National Cricket team,Shai Hope,Kamil Pooran,Arshdeep Singh,0.2,0.2,...,0,0,0,0,0,0,0,False,False,0.2
2,Ekana Cricket Stadium B Ground,1,1,West Indies Cricket Team,India National Cricket team,Shai Hope,Kamil Pooran,Arshdeep Singh,0.3,0.3,...,0,0,0,0,0,0,0,False,False,0.3
3,Ekana Cricket Stadium B Ground,1,1,West Indies Cricket Team,India National Cricket team,Shai Hope,Kamil Pooran,Arshdeep Singh,0.4,0.4,...,0,0,0,0,0,1,0,False,False,0.4
4,Ekana Cricket Stadium B Ground,1,1,West Indies Cricket Team,India National Cricket team,Shai Hope,Kamil Pooran,Arshdeep Singh,0.5,0.5,...,0,0,0,0,0,0,0,False,False,0.5


In [2]:
# Check column names
print("Column names:")
print(df.columns.tolist())

# Check missing values in important batting columns
batting_columns = [
    "Striker",
    "Batting Team",
    "Runs Off The Bat",
    "Wide",
    "No Ball",
    "Wicket",
    "4's",
    "6's",
    "Run_QC_Flag",
    "Extras_QC_Flag",
]

print("\nMissing values in batting columns:")
display(df[batting_columns].isnull().sum())

# Check whether scoring QC flags remain
print("\nRun QC issues:", df["Run_QC_Flag"].fillna(False).sum())
print("Extras QC issues:", df["Extras_QC_Flag"].fillna(False).sum())

Column names:
['Venue', 'MatchDay', 'Innings', 'Batting Team', 'Bowling Team', 'Striker', 'Non-Striker', 'Bowler', 'Ball Number', 'Actual Ball Number', 'Overs', 'Team Runs', 'Runs Off The Bat', 'Extra', 'Wide', 'No Ball', 'Leg Bye', 'Bye', 'Wicket', "4's", "6's", 'Run_QC_Flag', 'Extras_QC_Flag', 'Ball_Number_Label']

Missing values in batting columns:


Striker             0
Batting Team        0
Runs Off The Bat    0
Wide                0
No Ball             0
Wicket              0
4's                 0
6's                 0
Run_QC_Flag         0
Extras_QC_Flag      0
dtype: int64


Run QC issues: 0
Extras QC issues: 0


In [3]:
# Identify whether each delivery counts as a ball faced
df["Batter_Ball_Faced"] = (
    df["Wide"] == 0
).astype(int)

# Summarize batting performance by batter
batting_summary = (
    df.groupby("Striker", as_index=False)
    .agg(
        Runs=("Runs Off The Bat", "sum"),
        Balls_Faced=("Batter_Ball_Faced", "sum"),
        Fours=("4's", "sum"),
        Sixes=("6's", "sum"),
    )
)

# Calculate batting strike rate
batting_summary["Strike_Rate"] = (
    batting_summary["Runs"]
    / batting_summary["Balls_Faced"].replace(0, np.nan)
    * 100
).round(2)

# Sort by total runs
batting_summary = batting_summary.sort_values(
    "Runs", ascending=False
).reset_index(drop=True)

display(batting_summary.head(15))

,Striker,Runs,Balls_Faced,Fours,Sixes,Strike_Rate
0,Shai Hope,261,136,17,20,191.91
1,Sherfane Rutherford,221,118,19,15,187.29
2,Shreyas Iyer,159,74,15,9,214.86
3,Sanju Samson,152,79,16,8,192.41
4,Abhishek Sharma,139,76,14,8,182.89
5,Ishan Kishan,123,70,17,4,175.71
6,Tilak Varma,115,59,9,6,194.92
7,Shimron Hetmyer,81,41,11,4,197.56
8,Nitish Reddy,62,34,4,4,182.35
9,Kamil Pooran,59,40,6,3,147.50


In [4]:
# Inspect deliveries on which a wicket was recorded

wicket_records = df.loc[
    df["Wicket"] == 1,
    [
        "MatchDay",
        "Innings",
        "Batting Team",
        "Striker",
        "Non-Striker",
        "Bowler",
        "Ball Number",
        "Runs Off The Bat",
        "Extra",
        "Wicket",
    ],
].copy()

print("Total wicket records:", len(wicket_records))
display(wicket_records.head(20))

Total wicket records: 50


,MatchDay,Innings,Batting Team,Striker,Non-Striker,Bowler,Ball Number,Runs Off The Bat,Extra,Wicket
27,1,1,West Indies Cricket Team,Kamil Pooran,Shai Hope,Arshdeep Singh,4.3,0,0,1
32,1,1,West Indies Cricket Team,Shai Hope,Shimron Hetmyer,Axar Patel,5.2,0,0,1
35,1,1,West Indies Cricket Team,Rovman Powell,Shai Hope,Axar Patel,5.5,0,0,1
70,1,1,West Indies Cricket Team,Shai Hope,Sherfane Rutherford,Naman Dhir,11.4,0,0,1
75,1,1,West Indies Cricket Team,Roston Chase,Sherfane Rutherford,Kuldeep Yadav,12.3,0,0,1
80,1,1,West Indies Cricket Team,Romario Shepherd,Sherfane Rutherford,Naman Dhir,13.2,0,0,1
85,1,1,West Indies Cricket Team,Matthew Forde,Sherfane Rutherford,Kuldeep Yadav,14.1,0,0,1
95,1,1,West Indies Cricket Team,Shamar Springer,Sherfane Rutherford,Naman Dhir,15.5,0,0,1
113,1,1,West Indies Cricket Team,Sherfane Rutherford,Akeal Hosein,Mayank Yadav,18.5,0,0,1
115,1,1,West Indies Cricket Team,Akeal Hosein,Shamar Joseph,Mayank Yadav,19.1,0,0,1


In [5]:
print("All columns in the batting dataset:")
print(df.columns.tolist())

print("\nSample rows where a wicket occurred:")
display(
    df.loc[df["Wicket"] == 1, [
        "MatchDay",
        "Innings",
        "Striker",
        "Non-Striker",
        "Bowler",
        "Ball Number",
        "Wicket"
    ]].head(10)
)

All columns in the batting dataset:
['Venue', 'MatchDay', 'Innings', 'Batting Team', 'Bowling Team', 'Striker', 'Non-Striker', 'Bowler', 'Ball Number', 'Actual Ball Number', 'Overs', 'Team Runs', 'Runs Off The Bat', 'Extra', 'Wide', 'No Ball', 'Leg Bye', 'Bye', 'Wicket', "4's", "6's", 'Run_QC_Flag', 'Extras_QC_Flag', 'Ball_Number_Label', 'Batter_Ball_Faced']

Sample rows where a wicket occurred:


,MatchDay,Innings,Striker,Non-Striker,Bowler,Ball Number,Wicket
27,1,1,Kamil Pooran,Shai Hope,Arshdeep Singh,4.3,1
32,1,1,Shai Hope,Shimron Hetmyer,Axar Patel,5.2,1
35,1,1,Rovman Powell,Shai Hope,Axar Patel,5.5,1
70,1,1,Shai Hope,Sherfane Rutherford,Naman Dhir,11.4,1
75,1,1,Roston Chase,Sherfane Rutherford,Kuldeep Yadav,12.3,1
80,1,1,Romario Shepherd,Sherfane Rutherford,Naman Dhir,13.2,1
85,1,1,Matthew Forde,Sherfane Rutherford,Kuldeep Yadav,14.1,1
95,1,1,Shamar Springer,Sherfane Rutherford,Naman Dhir,15.5,1
113,1,1,Sherfane Rutherford,Akeal Hosein,Mayank Yadav,18.5,1
115,1,1,Akeal Hosein,Shamar Joseph,Mayank Yadav,19.1,1


In [6]:
# Calculate runs scored through boundaries
batting_summary["Boundary_Runs"] = (
    batting_summary["Fours"] * 4
    + batting_summary["Sixes"] * 6
)

# Calculate percentage of runs scored through boundaries
batting_summary["Boundary_Percentage"] = (
    batting_summary["Boundary_Runs"]
    / batting_summary["Runs"].replace(0, np.nan)
    * 100
).round(2)

# Display the top 15 batters by total runs
display(
    batting_summary[
        [
            "Striker",
            "Runs",
            "Balls_Faced",
            "Fours",
            "Sixes",
            "Strike_Rate",
            "Boundary_Runs",
            "Boundary_Percentage",
        ]
    ].head(15)
)

,Striker,Runs,Balls_Faced,Fours,Sixes,Strike_Rate,Boundary_Runs,Boundary_Percentage
0,Shai Hope,261,136,17,20,191.91,188,72.03
1,Sherfane Rutherford,221,118,19,15,187.29,166,75.11
2,Shreyas Iyer,159,74,15,9,214.86,114,71.70
3,Sanju Samson,152,79,16,8,192.41,112,73.68
4,Abhishek Sharma,139,76,14,8,182.89,104,74.82
5,Ishan Kishan,123,70,17,4,175.71,92,74.80
6,Tilak Varma,115,59,9,6,194.92,72,62.61
7,Shimron Hetmyer,81,41,11,4,197.56,68,83.95
8,Nitish Reddy,62,34,4,4,182.35,40,64.52
9,Kamil Pooran,59,40,6,3,147.50,42,71.19


In [7]:
# Identify dot balls: no batter runs and no extras
df["Dot_Ball"] = (
    (df["Runs Off The Bat"] == 0)
    & (df["Extra"] == 0)
    & (df["Wide"] == 0)
).astype(int)

# Count dot balls for each batter
dot_ball_summary = (
    df.groupby("Striker", as_index=False)
    .agg(Dot_Balls=("Dot_Ball", "sum"))
)

# Merge dot-ball counts into the batting summary
batting_summary = batting_summary.merge(
    dot_ball_summary,
    on="Striker",
    how="left"
)

# Calculate dot-ball percentage
batting_summary["Dot_Ball_Percentage"] = (
    batting_summary["Dot_Balls"]
    / batting_summary["Balls_Faced"].replace(0, np.nan)
    * 100
).round(2)

# Display the top 15 run-scorers
display(
    batting_summary.sort_values("Runs", ascending=False)[
        [
            "Striker",
            "Runs",
            "Balls_Faced",
            "Dot_Balls",
            "Dot_Ball_Percentage",
            "Strike_Rate",
        ]
    ].head(15)
)

,Striker,Runs,Balls_Faced,Dot_Balls,Dot_Ball_Percentage,Strike_Rate
0,Shai Hope,261,136,34,25.00,191.91
1,Sherfane Rutherford,221,118,35,29.66,187.29
2,Shreyas Iyer,159,74,11,14.86,214.86
3,Sanju Samson,152,79,19,24.05,192.41
4,Abhishek Sharma,139,76,22,28.95,182.89
5,Ishan Kishan,123,70,18,25.71,175.71
6,Tilak Varma,115,59,7,11.86,194.92
7,Shimron Hetmyer,81,41,13,31.71,197.56
8,Nitish Reddy,62,34,6,17.65,182.35
9,Kamil Pooran,59,40,13,32.50,147.50


In [8]:
# Check whether dot-ball counts are within the number of balls faced
batting_summary["Dot_Ball_Check"] = (
    batting_summary["Dot_Balls"] <= batting_summary["Balls_Faced"]
)

# Recalculate dot-ball percentage independently
batting_summary["Dot_Ball_Percentage_Check"] = (
    batting_summary["Dot_Balls"]
    / batting_summary["Balls_Faced"].replace(0, np.nan)
    * 100
).round(2)

print(
    "Batters where dot balls exceed balls faced:",
    (~batting_summary["Dot_Ball_Check"]).sum()
)

print(
    "Percentage mismatches:",
    (
        batting_summary["Dot_Ball_Percentage"]
        != batting_summary["Dot_Ball_Percentage_Check"]
    ).sum()
)

display(
    batting_summary[
        [
            "Striker",
            "Balls_Faced",
            "Dot_Balls",
            "Dot_Ball_Percentage",
            "Dot_Ball_Check",
            "Dot_Ball_Percentage_Check",
        ]
    ].head(10)
)

Batters where dot balls exceed balls faced: 0
Percentage mismatches: 0


,Striker,Balls_Faced,Dot_Balls,Dot_Ball_Percentage,Dot_Ball_Check,Dot_Ball_Percentage_Check
0,Shai Hope,136,34,25.00,True,25.00
1,Sherfane Rutherford,118,35,29.66,True,29.66
2,Shreyas Iyer,74,11,14.86,True,14.86
3,Sanju Samson,79,19,24.05,True,24.05
4,Abhishek Sharma,76,22,28.95,True,28.95
5,Ishan Kishan,70,18,25.71,True,25.71
6,Tilak Varma,59,7,11.86,True,11.86
7,Shimron Hetmyer,41,13,31.71,True,31.71
8,Nitish Reddy,34,6,17.65,True,17.65
9,Kamil Pooran,40,13,32.50,True,32.50


In [9]:
team_batting_summary = (
    df.groupby("Batting Team", as_index=False)
    .agg(
        Total_Runs=("Runs Off The Bat", "sum"),
        Balls_Faced=("Batter_Ball_Faced", "sum"),
        Fours=("4's", "sum"),
        Sixes=("6's", "sum"),
        Dot_Balls=("Dot_Ball", "sum"),
    )
)

# Calculate strike rate for each team
team_batting_summary["Strike_Rate"] = (
    team_batting_summary["Total_Runs"]
    / team_batting_summary["Balls_Faced"].replace(0, np.nan)
    * 100
).round(2)

# Calculate runs scored through boundaries
team_batting_summary["Boundary_Runs"] = (
    team_batting_summary["Fours"] * 4
    + team_batting_summary["Sixes"] * 6
)

team_batting_summary["Boundary_Percentage"] = (
    team_batting_summary["Boundary_Runs"]
    / team_batting_summary["Total_Runs"].replace(0, np.nan)
    * 100
).round(2)

team_batting_summary["Dot_Ball_Percentage"] = (
    team_batting_summary["Dot_Balls"]
    / team_batting_summary["Balls_Faced"].replace(0, np.nan)
    * 100
).round(2)

display(team_batting_summary)

,Batting Team,Total_Runs,Balls_Faced,Fours,Sixes,Dot_Balls,Strike_Rate,Boundary_Runs,Boundary_Percentage,Dot_Ball_Percentage
0,India National Cricket team,823,449,83,41,103,183.30,578,70.23,22.94
1,West Indies Cricket Team,839,469,69,57,144,178.89,618,73.66,30.70


In [10]:
# Compare total batter runs in the full dataset with team summary
print("Total runs off the bat in df:", df["Runs Off The Bat"].sum())

print(
    "Total runs off the bat in team summary:",
    team_batting_summary["Total_Runs"].sum()
)

print(
    "Total runs off the bat in batting summary:",
    batting_summary["Runs"].sum()
)

print("\nRows in df:", len(df))
print("Missing batter names:", df["Striker"].isna().sum())

Total runs off the bat in df: 1662
Total runs off the bat in team summary: 1662
Total runs off the bat in batting summary: 1662

Rows in df: 959
Missing batter names: 0


In [11]:
innings_runs_check = (
    df.groupby(
        ["MatchDay", "Innings", "Batting Team"],
        as_index=False
    )
    .agg(
        Runs_Off_Bat=("Runs Off The Bat", "sum"),
        Extras=("Extra", "sum"),
        Team_Runs=("Team Runs", "sum"),
        Deliveries=("Striker", "size"),
    )
)

display(innings_runs_check)

print(
    "Grand total — batter runs:",
    innings_runs_check["Runs_Off_Bat"].sum()
)

print(
    "Grand total — extras:",
    innings_runs_check["Extras"].sum()
)

print(
    "Grand total — team runs:",
    innings_runs_check["Team_Runs"].sum()
)

,MatchDay,Innings,Batting Team,Runs_Off_Bat,Extras,Team_Runs,Deliveries
0,1,1,West Indies Cricket Team,163,8,171,116
1,1,2,India National Cricket team,171,1,172,89
2,2,1,India National Cricket team,240,9,249,128
3,2,2,West Indies Cricket Team,248,4,252,115
4,3,1,India National Cricket team,213,10,223,124
5,3,2,West Indies Cricket Team,199,10,209,127
6,4,1,West Indies Cricket Team,229,18,247,130
7,4,2,India National Cricket team,199,24,223,130


Grand total — batter runs: 1662
Grand total — extras: 84
Grand total — team runs: 1746


In [12]:
match_batting_summary = (
    df.groupby(
        ["MatchDay", "Batting Team"],
        as_index=False
    )
    .agg(
        Runs_Off_Bat=("Runs Off The Bat", "sum"),
        Balls_Faced=("Batter_Ball_Faced", "sum"),
        Fours=("4's", "sum"),
        Sixes=("6's", "sum"),
        Dot_Balls=("Dot_Ball", "sum"),
        Extras=("Extra", "sum"),
        Team_Runs=("Team Runs", "sum"),
    )
)

match_batting_summary["Strike_Rate"] = (
    match_batting_summary["Runs_Off_Bat"]
    / match_batting_summary["Balls_Faced"].replace(0, np.nan)
    * 100
).round(2)

match_batting_summary["Dot_Ball_Percentage"] = (
    match_batting_summary["Dot_Balls"]
    / match_batting_summary["Balls_Faced"].replace(0, np.nan)
    * 100
).round(2)

display(match_batting_summary)

,MatchDay,Batting Team,Runs_Off_Bat,Balls_Faced,Fours,Sixes,Dot_Balls,Extras,Team_Runs,Strike_Rate,Dot_Ball_Percentage
0,1,India National Cricket team,171,89,19,7,19,1,172,192.13,21.35
1,1,West Indies Cricket Team,163,116,17,8,45,8,171,140.52,38.79
2,2,India National Cricket team,240,122,25,13,27,9,249,196.72,22.13
3,2,West Indies Cricket Team,248,111,21,20,34,4,252,223.42,30.63
4,3,India National Cricket team,213,120,24,8,23,10,223,177.50,19.17
5,3,West Indies Cricket Team,199,121,15,13,34,10,209,164.46,28.10
6,4,India National Cricket team,199,118,15,13,34,24,223,168.64,28.81
7,4,West Indies Cricket Team,229,121,16,16,31,18,247,189.26,25.62


In [13]:
# Check whether runs off bat + extras equals team runs
match_batting_summary["Runs_Reconciled"] = (
    match_batting_summary["Runs_Off_Bat"]
    + match_batting_summary["Extras"]
    == match_batting_summary["Team_Runs"]
)

display(
    match_batting_summary[
        [
            "MatchDay",
            "Batting Team",
            "Runs_Off_Bat",
            "Extras",
            "Team_Runs",
            "Balls_Faced",
            "Fours",
            "Sixes",
            "Dot_Balls",
            "Strike_Rate",
            "Dot_Ball_Percentage",
            "Runs_Reconciled",
        ]
    ].to_string(index=False)
)

print(
    "Innings with reconciliation issues:",
    (~match_batting_summary["Runs_Reconciled"]).sum()
)

' MatchDay                Batting Team  Runs_Off_Bat  Extras  Team_Runs  Balls_Faced  Fours  Sixes  Dot_Balls  Strike_Rate  Dot_Ball_Percentage  Runs_Reconciled\n        1 India National Cricket team           171       1        172           89     19      7         19       192.13                21.35             True\n        1    West Indies Cricket Team           163       8        171          116     17      8         45       140.52                38.79             True\n        2 India National Cricket team           240       9        249          122     25     13         27       196.72                22.13             True\n        2    West Indies Cricket Team           248       4        252          111     21     20         34       223.42                30.63             True\n        3 India National Cricket team           213      10        223          120     24      8         23       177.50                19.17             True\n        3    West Indies Cricket 

Innings with reconciliation issues: 0


In [14]:
# Create the final batting leaderboard
final_batting_leaderboard = batting_summary[
    [
        "Striker",
        "Runs",
        "Balls_Faced",
        "Strike_Rate",
        "Fours",
        "Sixes",
        "Boundary_Runs",
        "Boundary_Percentage",
        "Dot_Balls",
        "Dot_Ball_Percentage",
    ]
].copy()

# Sort by total runs scored
final_batting_leaderboard = (
    final_batting_leaderboard
    .sort_values("Runs", ascending=False)
    .reset_index(drop=True)
)

display(final_batting_leaderboard.head(20))

,Striker,Runs,Balls_Faced,Strike_Rate,Fours,Sixes,Boundary_Runs,Boundary_Percentage,Dot_Balls,Dot_Ball_Percentage
0,Shai Hope,261,136,191.91,17,20,188,72.03,34,25.00
1,Sherfane Rutherford,221,118,187.29,19,15,166,75.11,35,29.66
2,Shreyas Iyer,159,74,214.86,15,9,114,71.70,11,14.86
3,Sanju Samson,152,79,192.41,16,8,112,73.68,19,24.05
4,Abhishek Sharma,139,76,182.89,14,8,104,74.82,22,28.95
5,Ishan Kishan,123,70,175.71,17,4,92,74.80,18,25.71
6,Tilak Varma,115,59,194.92,9,6,72,62.61,7,11.86
7,Shimron Hetmyer,81,41,197.56,11,4,68,83.95,13,31.71
8,Nitish Reddy,62,34,182.35,4,4,40,64.52,6,17.65
9,Kamil Pooran,59,40,147.50,6,3,42,71.19,13,32.50


In [15]:
from pathlib import Path

OUTPUT_DIR = PROJECT_DIR / "Data" / "Processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = OUTPUT_DIR / "final_batting_leaderboard.csv"

final_batting_leaderboard.to_csv(OUTPUT_FILE, index=False)

print("Batting leaderboard saved successfully!")
print("File location:", OUTPUT_FILE)
print("Rows:", len(final_batting_leaderboard))
print("Columns:", len(final_batting_leaderboard.columns))

Batting leaderboard saved successfully!
File location: C:\India_Vs_West_Indies_T20i_Challenge\Data\Processed\final_batting_leaderboard.csv
Rows: 26
Columns: 10
